<a href="https://colab.research.google.com/github/naubergois/curso-segmentacao-colab/blob/main/notebooks/02-segmentacao-em-video.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 2. Segmentação quadro a quadro

Vídeo é uma sequência de fotos. O laço lê o quadro, roda o modelo, desenha e grava. A diferença é o que se conta. Um operário que atravessa a obra gera uma detecção por quadro, mas a pergunta melhor é quantos pixels a pessoa ocupa.

O clipe é uma obra de rodovia: gente de colete e máquinas no aterro. Dura uns 24 segundos. Aqui são 80 quadros, para caber em CPU.

A escavadeira não está nas 80 classes. O nome que volta costuma ser `train` ou `truck`. A conta de pixels continua valendo, com o nome que o modelo conhece.

> GPU deixa bem mais rápido: **Ambiente de execução → Alterar o tipo de ambiente → T4 GPU**.

In [ ]:
!pip install -q ultralytics

In [ ]:
import base64
import subprocess
import urllib.request
from pathlib import Path

from IPython.display import HTML, display


def baixar(url, destino):
    if not Path(destino).exists():
        urllib.request.urlretrieve(url, destino)
    print(f"{destino}: {Path(destino).stat().st_size / 1e6:.1f} MB")
    return destino


def mascaras_originais(resultado):
    """Máscaras no tamanho da foto, como array booleano (n, altura, largura).

    A rede responde numa grade menor que a imagem. Aqui a máscara é esticada
    de volta ao tamanho original, para a área bater com os pixels da foto.
    """
    import cv2
    import numpy as np

    altura, largura = resultado.orig_shape
    if resultado.masks is None or len(resultado.masks) == 0:
        return np.zeros((0, altura, largura), dtype=bool)
    saida = []
    for mascara in resultado.masks.data.cpu().numpy():
        mascara = cv2.resize(mascara, (largura, altura), interpolation=cv2.INTER_NEAREST)
        saida.append(mascara > 0.5)
    return np.stack(saida)


def mostrar_video(caminho, largura=720):
    """Converte para H.264 (o navegador não toca mp4v) e embute no notebook."""
    web = str(Path(caminho).with_suffix("")) + "_web.mp4"
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", caminho,
         "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart", web],
        check=True,
    )
    dados = base64.b64encode(Path(web).read_bytes()).decode()
    display(HTML(f'<video width="{largura}" controls autoplay muted loop>'
                 f'<source src="data:video/mp4;base64,{dados}" type="video/mp4"></video>'))
    return web


def baixar_para_maquina(caminho):
    try:
        from google.colab import files
        files.download(caminho)
    except ImportError:
        print(f"Fora do Colab: o arquivo está em {Path(caminho).resolve()}")

In [ ]:
from collections import Counter

import cv2
from ultralytics import YOLO

model = YOLO("yolov8n-seg.pt")
video = baixar(
    "https://raw.githubusercontent.com/naubergois/curso-segmentacao-colab/main/midia/obra.mp4",
    "obra.mp4",
)

## Ler, segmentar, gravar

`r.plot()` já pinta as máscaras. Em paralelo, somamos os pixels de cada classe e guardamos os nomes que apareceram em cada quadro.

In [ ]:
MAX_QUADROS = 80

cap = cv2.VideoCapture(video)
fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25
largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
altura = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"{largura}x{altura} a {fps} FPS")

out = cv2.VideoWriter("obra_segmentada.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (largura, altura))
pixels = Counter()
classes_por_quadro = []
mascaras_por_quadro = []
quadros = 0

while quadros < MAX_QUADROS:
    ok, frame = cap.read()
    if not ok:
        break
    r = model(frame, verbose=False)[0]
    out.write(r.plot())
    nomes = []
    if r.masks is not None:
        for c, m in zip(r.boxes.cls.int().tolist(), mascaras_originais(r)):
            nome = model.names[c]
            nomes.append(nome)
            pixels[nome] += int(m.sum())
    classes_por_quadro.append(nomes)
    mascaras_por_quadro.append(len(nomes))
    quadros += 1

cap.release()
out.release()
print(f"{quadros} quadros processados")
assert quadros > 0

## Assistir o resultado

In [ ]:
web = mostrar_video("obra_segmentada.mp4")

## Quem ocupa a imagem

O número de detecções trata um operário pequeno e uma máquina enorme como iguais. A soma de pixels não.

In [ ]:
for classe, n in pixels.most_common():
    print(f"{classe:16s} {n:8d} pixels")

## Baixar o vídeo

In [ ]:
baixar_para_maquina(web)

## Exercícios

Seis perguntas sobre o que acabou de rodar. Escreva a resposta na variável `resposta` e rode a célula: a função `conferir` diz na hora se está certo. Se travar, abra **Ver solução**.

Rode antes a célula de preparação abaixo. Ela só carrega as verificações.

In [ ]:
# @title Preparação dos exercícios (rode, não precisa ler)
from functools import lru_cache
from pathlib import Path

_cap = cv2.VideoCapture(video)
_total = int(_cap.get(cv2.CAP_PROP_FRAME_COUNT))
_fps_arquivo = _cap.get(cv2.CAP_PROP_FPS)
_cap.release()


@lru_cache(None)
def _pixels_pessoa_pulando():
    cap = cv2.VideoCapture(video)
    total = 0
    for i in range(40):
        ok, frame = cap.read()
        if not ok:
            break
        if i % 5:
            continue
        r = model(frame, verbose=False)[0]
        if r.masks is None:
            continue
        for c, m in zip(r.boxes.cls.int().tolist(), mascaras_originais(r)):
            if model.names[c] == "person":
                total += int(m.sum())
    cap.release()
    return total


VERIFICAR = {
    1: lambda r: abs(float(r) - _total / _fps_arquivo) < 0.05,
    2: lambda r: r in {nome for nome, n in pixels.items() if n == max(pixels.values())},
    3: lambda r: int(r) == sum("person" in nomes for nomes in classes_por_quadro[:40]),
    4: lambda r: int(r) == _pixels_pessoa_pulando(),
    5: lambda r: abs(float(r) - (sum(mascaras_por_quadro) / len(mascaras_por_quadro))) < 1e-3,
    6: lambda r: Path(str(r)).name == "obra_pessoas.mp4" and Path(str(r)).stat().st_size > 1000,
}

_acertos = {}


def conferir(n, resposta):
    if resposta is None:
        print(f"Exercício {n}: ainda sem resposta.")
        return
    try:
        certo = bool(VERIFICAR[n](resposta))
    except Exception as erro:
        certo = False
        print(f"  (a verificação deu erro: {erro})")
    _acertos[n] = certo
    print(f"Exercício {n}: " + ("✅ certo!" if certo else "❌ ainda não. Releia o enunciado e tente de novo."))


def placar():
    print(f"{sum(_acertos.values())} de {len(VERIFICAR)} exercícios certos.")

### Exercício 1. Duração do vídeo

Quantos segundos tem o vídeo inteiro? Use o total de quadros e o FPS do arquivo, não só os quadros que a aula processou.

<details>
<summary>Ver solução</summary>

```python
cap = cv2.VideoCapture(video)
resposta = cap.get(cv2.CAP_PROP_FRAME_COUNT) / cap.get(cv2.CAP_PROP_FPS)
cap.release()
```

</details>

In [ ]:
# Dica: cv2.CAP_PROP_FRAME_COUNT e cv2.CAP_PROP_FPS.
resposta = None

conferir(1, resposta)

### Exercício 2. Quem ocupa mais pixels

Qual classe somou mais pixels de máscara no vídeo processado? Responda com o nome.

<details>
<summary>Ver solução</summary>

```python
resposta = pixels.most_common(1)[0][0]
```

</details>

In [ ]:
# Dica: pixels é um Counter: pixels.most_common(1) devolve o par (nome, total).
resposta = None

conferir(2, resposta)

### Exercício 3. Quadros com pessoa

Em quantos dos primeiros 40 quadros processados aparece pelo menos uma pessoa?

<details>
<summary>Ver solução</summary>

```python
resposta = sum("person" in nomes for nomes in classes_por_quadro[:40])
```

</details>

In [ ]:
# Dica: classes_por_quadro[i] é a lista de nomes daquele quadro.
resposta = None

conferir(3, resposta)

### Exercício 4. Pular quadros

Nos primeiros 40 quadros, rode o modelo só em 1 a cada 5 (quadros 0, 5, 10...) e some os pixels das máscaras de `person`.

<details>
<summary>Ver solução</summary>

```python
cap = cv2.VideoCapture(video)
total = 0
for i in range(40):
    ok, frame = cap.read()
    if not ok:
        break
    if i % 5:
        continue
    r = model(frame, verbose=False)[0]
    if r.masks is None:
        continue
    for c, m in zip(r.boxes.cls.int().tolist(), mascaras_originais(r)):
        if model.names[c] == "person":
            total += int(m.sum())
cap.release()
resposta = total
```

</details>

In [ ]:
# Dica: if i % 5: continue pula o quadro. mascaras_originais(r) devolve as máscaras no tamanho da foto.
resposta = None

conferir(4, resposta)

### Exercício 5. Média de máscaras

Em média, quantas máscaras o modelo desenhou por quadro no vídeo processado?

<details>
<summary>Ver solução</summary>

```python
resposta = sum(mascaras_por_quadro) / len(mascaras_por_quadro)
```

</details>

In [ ]:
# Dica: mascaras_por_quadro guarda a contagem de cada quadro.
resposta = None

conferir(5, resposta)

### Exercício 6. Vídeo só com pessoas

Grave um vídeo novo com os primeiros 40 quadros, desenhando só `person`. A resposta é o caminho `obra_pessoas.mp4`. O índice da classe está em `model.names`.

<details>
<summary>Ver solução</summary>

```python
indice = [i for i, nome in model.names.items() if nome == "person"][0]
cap = cv2.VideoCapture(video)
fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25
largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
altura = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter("obra_pessoas.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (largura, altura))
for _ in range(40):
    ok, frame = cap.read()
    if not ok:
        break
    r = model(frame, classes=[indice], verbose=False)[0]
    out.write(r.plot())
cap.release()
out.release()
resposta = "obra_pessoas.mp4"
```

</details>

In [ ]:
# Dica: Passe classes=[indice] na chamada do modelo e grave com r.plot().
resposta = None

conferir(6, resposta)

In [ ]:
placar()

## O que fica

Segmentar vídeo é o mesmo laço da detecção, com uma conta a mais: a área. Para desenhar só uma classe, passe `classes=[indice]` na chamada do modelo. O índice está em `model.names`.